In [1]:
import torch
import testdata
# note: had to move this notebook and testdata.py into 
# the multicor_fa directory to run
from _em import _EM_step_no_private_stable, fit_EM_iter

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

Generate fake data

In [6]:
params = {
    'd': 15, 
    'k': [0, 0, 0], 
    'p': [15, 13, 8], 
    'n': 5000,
    'sigsq': [0.3, 0.7, 0.5]
}

Y, W, L, Phi = testdata.simulate_data(params, private_var=False, verbose=True)
W_init, L_init, Phi_init = testdata.initialize_params(W, L, Phi, private_var=False)

# need Y to be N x p_all
Y = Y.T

No private factors, so Y = WZ + E


Attempt at more systematic testing for complete data case

In [30]:
metrics = {
    'WWt_corr': [],
    'Phi_corr': [],
}
problematic_runs = []
# simulate 10000 runs 
for i in range(10000):
    if (i+1) % 2000 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]
    
    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_runs.append(phi_corr)
    metrics['Phi_corr'].append(phi_corr)

2000 simulations completed
4000 simulations completed
6000 simulations completed
8000 simulations completed
10000 simulations completed


Manual note: 
- 1000 simulations with complete data took ~ 3m 29.8s to run
- 10000 simultations with complete data took ~ 36m 40.2s to run


In [31]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9932
	Min: 0.965
	Max: 0.9981
Phi_corr
	Mean: 0.9865
	Min: -0.2531
	Max: 0.9994


In [36]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {len(problematic_runs)/10000 * 100}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 1.18%


Attempt at more systematic testing for missing data case

In [37]:
metrics = {
    'WWt_corr': [],
    'Phi_corr': [],
}
problematic_runs = []
# simulate 1000 runs 
for i in range(10000):
    if (i+1) % 2000 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    # insert missing data
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')
    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    # need to fill in NA Sigma values here
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_runs.append(phi_corr)
    metrics['Phi_corr'].append(phi_corr)

2000 simulations completed
4000 simulations completed
6000 simulations completed
8000 simulations completed
10000 simulations completed


Manual note: 
- 1000 simulations with missing data took ~ 2m 26.3s
- 10000 simulations with missing data took ~ 25m 32.8s

In [38]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9967
	Min: 0.9678
	Max: 0.9987
Phi_corr
	Mean: 0.9813
	Min: -0.1639
	Max: 0.999


In [40]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {round(len(problematic_runs)/10000 * 100, 2)}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 1.89%


Missing data with more than one mode missing in some samples

In [41]:
metrics = {
    'WWt_corr': [],
    'Phi_corr': [],
}
problematic_runs = []
# simulate 1000 runs 
for i in range(10000):
    if (i+1) % 2000 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    # insert missing data
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1020:1070, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1065:1100, params['p'][0]+params['p'][1]:] = float('nan')
    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    # need to fill in NA Sigma values here
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_runs.append(phi_corr)
    metrics['Phi_corr'].append(phi_corr)

2000 simulations completed
4000 simulations completed
6000 simulations completed
8000 simulations completed
10000 simulations completed


1000 simulations ran in 2m 26.9s

In [42]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9967
	Min: 0.9697
	Max: 0.999
Phi_corr
	Mean: 0.9811
	Min: -0.2005
	Max: 0.9991


In [43]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {round(len(problematic_runs)/10000 * 100, 2)}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 2.15%
